I need to put this notebook here:
/global/scratch/users/empchase/EC_libtest/analysis
so i can check whether the BCs match the A10 union map

Question to answer: “Are the number of RPTR BCs detected here similar to the number detected in cDNA? Newest Maxipreps?”

In [1]:
import datetime
import pandas as pd
import re
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import csv

import duckdb

conn = duckdb.connect('/global/scratch/users/empchase/Seq_analysis/mydb.db') # create an in-memory database

In [2]:
pwd

'/global/scratch/users/empchase/EC_libtest/analysis'

In [3]:
path = '/global/scratch/users/empchase/EC_libtest/analysis/'

RPTR1f = path + 'Staller_R1_MVS_0095_I1_TGCAGGCTAT_GTTCACTTAG_S209.fastq.gz.assembled.fastq'
RPTR2f = path + 'Staller_R2_MVS_0094_I1_TGACGTCGTC_GATAAGTACG_S210.fastq.gz.assembled.fastq'
RPTR1files = [RPTR1f, RPTR2f]

In [4]:
rc1 = 'ACTCGAG' #7bp ; before rptr barcode
rc2 = 'GGCCGCt' #7 bp ; after rptr barcode

In [5]:
def getmid(seq, pre, post, bclen):
    # seq = the sequence to parse
    # pre = substring that precedes piece of interest
    # post = substring that follows piece of interest
    # returns piece of interest

    re_key = pre + "(.*)"+ post 
    poi_search = re.search(re_key, seq)
#     print(poi_search)
#     poi = ''


    if poi_search is None:
        #the barcode will be called X
        poi = "X"
        
        #then we search for which restriction site had the error
        #regex for the bc we want to ignore
        w = "(.{"+str(bclen)+"})" 
#         print(w)
        pre_re = pre + w + "(.{7})"
        pre_search = re.search(pre_re, seq)
#         print(pre_search)
        post_re = "(\w{7})" + w + post
        post_search = re.search(post_re, seq)
#         print(post_search)
        
        if pre_search is None and post_search is None:
            a = 'X'
            z = 'X'
        elif pre_search is None:
            poi = post_search.group(2)
            a = post_search.group(1)
            z = post
        elif post_search is None:
            poi = pre_search.group(1)
            z = pre_search.group(2)
            a = pre
        else:
            a = "Z"
            z = 'Z'
#             print(seq)
            
    else:
        poi = poi_search.group(1)
        a = pre
        z = post
    
    return poi, a, z



# function that just looks for bc
def rbc_finder(readfile, bc_pre=rc1, bc_post=rc2, bc_len=14):

    # make lists of reads
    seqlist = []
    
    with open(readfile, 'r') as fin:
        for line in fin:
            if line.startswith('@'):
                #look at next line to get read sequence, add to list
                seq = next(fin)
                seq = seq.strip()
                seqlist.append(seq)

    
    #make lists of BCs from list of reads
    bc_list = []
    bc_lens = []
#     q_list = []
    
#     prex_list = []
#     prex_len = []
#     postx_list = []
#     postx_len = []
       
    for read in seqlist:
   
        bc, prex, postx = getmid(read, bc_pre, bc_post, bc_len)
        bc_list.append(bc)
        bcl = len(bc)
        bc_lens.append(bcl)
        
#         prex_list.append(prex)
#         prex_len.append(len(prex))
#         postx_list.append(postx)
#         postx_len.append(len(postx))
#         if bcl == bc_len:
#             q_list.append(1)
#         else:
#             q_list.append(0)
    #         print(bc)

    
    BC_dict = {"BCs":bc_list, "Length":bc_lens #,
#                "Qual":q_list, "prex":prex_list, 'prexl':prex_len,
#                'postx':postx_list, 'postxl':postx_len
              } 
    BC_df = pd.DataFrame.from_dict(BC_dict)
    
    #label BC with library name
    
    libname = '_'.join(readfile.split('/')[-1].split('_')[0:6])
    BC_df['Library'] = libname
    
    return BC_df

In [6]:
t1r = rbc_finder(RPTR1files[0])
t1r

,BCs,Length,Library
0,TACCAGATGTCGAG,14,Staller_R1_MVS_0095_I1_TGCAGGCTAT
1,TAAAAATGCCTAAT,14,Staller_R1_MVS_0095_I1_TGCAGGCTAT
2,AAGGACGTCTGGTC,14,Staller_R1_MVS_0095_I1_TGCAGGCTAT
3,X,1,Staller_R1_MVS_0095_I1_TGCAGGCTAT
4,X,1,Staller_R1_MVS_0095_I1_TGCAGGCTAT
...,...,...,...
198397,GGGAGACGATAATG,14,Staller_R1_MVS_0095_I1_TGCAGGCTAT
198398,TCCTCCCTCCACCA,14,Staller_R1_MVS_0095_I1_TGCAGGCTAT
198399,GTGGAAAACAATGA,14,Staller_R1_MVS_0095_I1_TGCAGGCTAT
198400,ACCCTTTAAGTCGA,14,Staller_R1_MVS_0095_I1_TGCAGGCTAT


In [7]:
t2r = rbc_finder(RPTR1files[1])
t2r

,BCs,Length,Library
0,AGTGTATCCCAGGC,14,Staller_R2_MVS_0094_I1_TGACGTCGTC
1,TTGGCACTACTAAC,14,Staller_R2_MVS_0094_I1_TGACGTCGTC
2,GGGCCATTGAGTAC,14,Staller_R2_MVS_0094_I1_TGACGTCGTC
3,CCCCTACGTGAAGT,14,Staller_R2_MVS_0094_I1_TGACGTCGTC
4,GGGTCATCTATTAA,14,Staller_R2_MVS_0094_I1_TGACGTCGTC
...,...,...,...
213055,GTGTATCTTCCTTT,14,Staller_R2_MVS_0094_I1_TGACGTCGTC
213056,TCAAACTTGCTTAC,14,Staller_R2_MVS_0094_I1_TGACGTCGTC
213057,CACACGCGTTGTTT,14,Staller_R2_MVS_0094_I1_TGACGTCGTC
213058,TAATCAGAGTCCAG,14,Staller_R2_MVS_0094_I1_TGACGTCGTC


In [8]:
#read in as df and create dictionary
tbbdf = conn.sql("""SELECT DISTINCT * FROM TBB""").df() #maybe do select distinct next time
tbbdf.head()

,Tiles,ADbc,RPTRbc,TBBs,TILEADBC
0,GGTAGAAGGAAATCTGGTAATTCTCCACCATCTACTAGACATTCTT...,GCAAATCTATC,TACCCGAAGCCGAA,GGTAGAAGGAAATCTGGTAATTCTCCACCATCTACTAGACATTCTT...,GGTAGAAGGAAATCTGGTAATTCTCCACCATCTACTAGACATTCTT...
1,CATTCTTCTGTTGCTGGTGTTAATGCTAGGAAGAGAGATAAACCAT...,TGACCTTTGGC,AGTAATTCGGAAGC,CATTCTTCTGTTGCTGGTGTTAATGCTAGGAAGAGAGATAAACCAT...,CATTCTTCTGTTGCTGGTGTTAATGCTAGGAAGAGAGATAAACCAT...
2,ACTATTCCAGAAGATTGGTATCCATTGTTTCCATCTATGGGTGATG...,CGTACCAATGA,CAAGCGCTTCTCAC,ACTATTCCAGAAGATTGGTATCCATTGTTTCCATCTATGGGTGATG...,ACTATTCCAGAAGATTGGTATCCATTGTTTCCATCTATGGGTGATG...
3,GATTCTGCTCAATTGGATTCTAATATGGAAGCTGAAATTGAAGCTA...,CAACGAGCAAC,AATCCAGTCCATAG,GATTCTGCTCAATTGGATTCTAATATGGAAGCTGAAATTGAAGCTA...,GATTCTGCTCAATTGGATTCTAATATGGAAGCTGAAATTGAAGCTA...
4,CCAGCTCCAGCTGTTGCTCCACAAGCTTCTCCAGTTGTTGCTTCTT...,ACATAAACGGG,GTGCAGCTTCGTGA,CCAGCTCCAGCTGTTGCTCCACAAGCTTCTCCAGTTGTTGCTTCTT...,CCAGCTCCAGCTGTTGCTCCACAAGCTTCTCCAGTTGTTGCTTCTT...


In [9]:
tbbdf.shape

(28241827, 5)

In [10]:
# tilecov_tbbdf = tbbdf['Tiles'].value_counts().to_frame().reset_index()
# print(tilecov_tbbdf.shape)
# tilecov_tbbdf.head()

(20695, 2)


,index,Tiles
0,TTGACTGCTTTGACTTCTCCATCTTTGTTTGATGGTTCTCCAGACT...,5801
1,TCTACTGATTCTACTCCAATGTTTGAATATGAGAATTTGGAAGATA...,3434
2,GATGTTACTCCAGCTTCTCCAATGATTAGAACTACTTCTTCTAGAG...,3341
3,TCTACTGCTCCACCAACTGATGTTTCTTTGGGTGATGAATTGCATT...,3290
4,GATTCTCCAGGTCCAGGATTTACTCCACATGATTCTGCTCCATATG...,3167


In [ ]:
# rptrcov_tbbdf = tbbdf['RPTRbc'].value_counts().to_frame().reset_index()
# print(rptrcov_tbbdf.shape)
# rptrcov_tbbdf.head()

In [11]:
# sns.histplot(data=tilecov_tbbdf, x="Tiles")
# plt.hist(tilecov_tbbdf["Tiles"])
# plt.show()

#failed to map segment from shared object: Cannot allocate memory

ImportError: /global/home/users/empchase/.conda/envs/biopython/lib/python3.11/site-packages/matplotlib/backends/_backend_agg.cpython-311-x86_64-linux-gnu.so: failed to map segment from shared object: Cannot allocate memory

In [9]:
#dictionary ~ {RPTRbc: TBB}

RPTRdict = dict(zip(tbbdf.RPTRbc, tbbdf.TBBs))

In [10]:
def analyze_bcs (df, bc_len, bcdict):
    # df = barcode containing df, parsed from fastq
    # bc_len = int, expected barcode length
    # tbb_dictkey = str, either 'ADbc' or 'RPTRbc'
    
    print(df.loc[0,'Library'])
    print()
    print('total reads')
    tr = df.shape[0]
    print(tr) #how many total reads
    
    cls = df[df['Length']== bc_len] #cl = correct length
    print('BCs of correct length')
    clcount = cls.shape[0]
    print(clcount)
    
    print('% BCs of correct length')
    clpct = cls.shape[0]/df.shape[0]
    print (clpct)
    
    #df of BC coverage
    cl_covdf = cls['BCs'].value_counts().to_frame().reset_index() 
    print('# Unique BCs')
    uniqbccount = cl_covdf.shape[0]
    print(uniqbccount)
#     print('SUM Unique BCs')
#     print(cl_covdf.sum(numeric_only=True)['BCs'])

    #copy down unique BCs into a list
    bcs = cl_covdf['index'].tolist()
    


    matchlist = [] #list of TBBs that match the BCs
#     matches = 0
    for x in bcs:
        try:
            tbb = bcdict[x]
        except KeyError:
            matchlist.append(0)
        else:
            matchlist.append(tbb)
#             matches += 1
            

            
    cl_covdf['TBB'] = matchlist
    
    matchesonly = cl_covdf.replace(0, np.nan)
    matchesonly = matchesonly.dropna()
    totmatches = matchesonly.sum(numeric_only=True)['BCs']
    print('# BC matches to A10 deep seq map')
    mapmatches = matchesonly.shape[0]
    print(mapmatches)
    
    print('TOT BC matches to A10 deep seq map')
    print(totmatches)
    
#     print('% unique BCs matched')
#     print(totmatches/cl_covdf.shape[0])
    
    
    print()
    
    #label the df with library name
    
    libname = df.loc[0,'Library']
    cl_covdf['Library'] = libname
    
    statslist = [libname, tr, clcount, clpct, uniqbccount, mapmatches, totmatches]
    
    
    return cl_covdf, statslist
    

In [11]:
def split_TBB (countsdf):
    # finds the putative tiles and ADs via the TBB and adds it to the df IN PLACE
    putative_tiles = []
    putative_ads = []

    for index, row in countsdf.loc[:, ['TBB']].iterrows():
        try:
            #     print(row['TBB'])
            tbb_str = row['TBB']
            ptile = tbb_str.split('-')[0]
            pad = tbb_str.split('-')[1]
        except AttributeError:
            ptile = 0
            pad = 0 
        finally:
            putative_tiles.append(ptile)
            putative_ads.append(pad)

    countsdf['Putative_Tile'] = putative_tiles
    countsdf['Putative_AD'] = putative_ads

In [12]:
Rlist = [t1r, t2r]

In [19]:
#analyze all dfs and get dictionary of ADs:TBBs from each df
#the dfs we keep in the end in analyzed_rptr_dfs are not clean (have 0s and RPTR BCs not matching to A10)

analyzed_rptr_dfs = []
ADdict = {}
rptrstats = []

for og_df in Rlist:
    # get the counts dfs, split the TBBs in each
    counts_df, statslist = analyze_bcs(og_df, 14, RPTRdict)
    rptrstats.append(statslist)
#     print(len(rptrstats))
    split_TBB(counts_df)
#     analyzed_rptr_dfs.append(counts_df)
    
    counts_df_clean = counts_df.replace(0, np.nan)
    counts_df_clean = counts_df_clean.dropna()
    analyzed_rptr_dfs.append(counts_df_clean)

    ad_dict = dict(zip(counts_df_clean.Putative_AD, counts_df_clean.TBB))
    ADdict.update(ad_dict)

Staller_R1_MVS_0095_I1_TGCAGGCTAT

total reads
198402
BCs of correct length
184190
% BCs of correct length
0.9283676575840969
# Unique BCs
80564
# BC matches to A10 deep seq map
54282
TOT BC matches to A10 deep seq map
117164

Staller_R2_MVS_0094_I1_TGACGTCGTC

total reads
213060
BCs of correct length
199194
% BCs of correct length
0.9349197409180513
# Unique BCs
84620
# BC matches to A10 deep seq map
56994
TOT BC matches to A10 deep seq map
126685



In [14]:
rstats_tocsv = pd.DataFrame(rptrstats, columns = [
    'Library', 'TotalReads', 'Reads_correctlen', 'Reads_clpct', 'TotBC_Unique','MatchesCounts_Unique', 'MatchesCounts_Total'])

rstats_tocsv.head() #this will tell us if TBB df has changed with distinct    

,Library,TotalReads,Reads_correctlen,Reads_clpct,TotBC_Unique,MatchesCounts_Unique,MatchesCounts_Total
0,Staller_R1_MVS_0095_I1_TGCAGGCTAT,198402,184190,0.928368,80564,54282,117164
1,Staller_R2_MVS_0094_I1_TGACGTCGTC,213060,199194,0.934920,84620,56994,126685


In [15]:
len(ADdict) #for experimental: 6040 in TBB_Union (the misnomer); 9409 in TBB (the real union table)

82819

In [26]:
# rstats_tocsv.to_csv('ECLibtest_RPTRstats_TBB.csv')

In [16]:
ADfile = path + 'Staller_TF_MVS_0096_I1_GAACAAGCGG_TAGCGGAATT_S208.fastq.gz.assembled.fastq'
ad1 = 'GCGCGCC'
ad2 = 'GGGCCCG'
t1a = rbc_finder(ADfile, bc_pre=ad1, bc_post=ad2, bc_len=11)
t1a.head()

,BCs,Length,Library
0,TGTAAGTTTCA,11,Staller_TF_MVS_0096_I1_GAACAAGCGG
1,TTACGCCAGGA,11,Staller_TF_MVS_0096_I1_GAACAAGCGG
2,TTACGCCAGGA,11,Staller_TF_MVS_0096_I1_GAACAAGCGG
3,TTTTAGTAACC,11,Staller_TF_MVS_0096_I1_GAACAAGCGG
4,TGTAAGTTTCA,11,Staller_TF_MVS_0096_I1_GAACAAGCGG


In [17]:
#analyze all dfs and get dictionary of ADs:TBBs from each df
#the dfs we keep in the end in analyzed_rptr_dfs are not clean (have 0s and RPTR BCs not matching to A10)


# get the counts dfs, split the TBBs in each
analyzed_ad_df, adstats = analyze_bcs(t1a, 11, ADdict)
split_TBB(counts_df)

adcounts_df_clean = analyzed_ad_df.replace(0, np.nan)
adcounts_df_clean = analyzed_ad_df.dropna()



Staller_TF_MVS_0096_I1_GAACAAGCGG

total reads
369602
BCs of correct length
369051
% BCs of correct length
0.9985092072012597
# Unique BCs
3610
# BC matches to A10 deep seq map
912
TOT BC matches to A10 deep seq map
125371



In [20]:
analyzed_rptr_dfs[0]

,index,BCs,TBB,Library,Putative_Tile,Putative_AD
2,CCTGACTGATAGGG,840,AGAGATAAACCATTGCCACCAATTGTTGTTGAAGATCCAAATGATA...,Staller_R1_MVS_0095_I1_TGCAGGCTAT,AGAGATAAACCATTGCCACCAATTGTTGTTGAAGATCCAAATGATA...,ACTAGCTTTTA
3,ATACCCTTCGCTAG,599,CAACCAATCTTGGATCCACAAACTGCTGCTGCTTACTTTAAGAATT...,Staller_R1_MVS_0095_I1_TGCAGGCTAT,CAACCAATCTTGGATCCACAAACTGCTGCTGCTTACTTTAAGAATT...,CTGCGCTAAGT
4,GAGGTGAAATGTTA,212,TCTAATAAGATTACTACTTTGGATTTGCCAGTTGTTACTACTGATT...,Staller_R1_MVS_0095_I1_TGCAGGCTAT,TCTAATAAGATTACTACTTTGGATTTGCCAGTTGTTACTACTGATT...,TCTACGCTTTC
5,GTCCTTTATTACAC,175,AATCAACAACCACCAGCTAAGATGGATTTGCAAGATGCATTCTCTT...,Staller_R1_MVS_0095_I1_TGCAGGCTAT,AATCAACAACCACCAGCTAAGATGGATTTGCAAGATGCATTCTCTT...,TGTACAGATGC
7,ATATAACCGAACTT,135,TCTCAACATAGATTGTCTACTCCATCTGTTCCACCAGTTCCATTGT...,Staller_R1_MVS_0095_I1_TGCAGGCTAT,TCTCAACATAGATTGTCTACTCCATCTGTTCCACCAGTTCCATTGT...,ATATATAATAA
...,...,...,...,...,...,...
80559,TCCTAGCCACAAAG,1,TTGTTTGATTTGCCATTGGACTTTGGTGCTGATGATGTTACTAAAG...,Staller_R1_MVS_0095_I1_TGCAGGCTAT,TTGTTTGATTTGCCATTGGACTTTGGTGCTGATGATGTTACTAAAG...,TTTACAGAGTT
80560,GTATCCTTAACGTG,1,TCTTCTTCATTCTCTTCTGCTTCTGCTAATATCTTGCCAAGAGACT...,Staller_R1_MVS_0095_I1_TGCAGGCTAT,TCTTCTTCATTCTCTTCTGCTTCTGCTAATATCTTGCCAAGAGACT...,AATGAGTCGTG
80561,GAGGTAAACCGGGG,1,TCTCAATCTACTTGGTTTCCATCTTCTACTCCATCTTTGCCAGCTT...,Staller_R1_MVS_0095_I1_TGCAGGCTAT,TCTCAATCTACTTGGTTTCCATCTTCTACTCCATCTTTGCCAGCTT...,AATAGACTTGA
80562,TGATTCGGGCGATC,1,TCTACTTCTTCTTTAAATTCTCCAGAATCTACTATTGCTACTACTG...,Staller_R1_MVS_0095_I1_TGCAGGCTAT,TCTACTTCTTCTTTAAATTCTCCAGAATCTACTATTGCTACTACTG...,CACTGTTGAAC


In [21]:
sns.histplot(data=analyzed_rptr_dfs[0], x="BCs")
plt.show()



ImportError: /global/home/users/empchase/.conda/envs/biopython/lib/python3.11/site-packages/matplotlib/backends/_backend_agg.cpython-311-x86_64-linux-gnu.so: failed to map segment from shared object: Cannot allocate memory

In [23]:
import pyarrow

In [25]:
tbbdfname = 'trueUniondf.parquet.gzip'
lib1 = 'EClibtest_RPTR1df.csv'
lib2 = 'EClibtest_RPTR2df.csv'
lib3 = 'EClibtest_ADdf.csv'

# tbbdf.to_parquet(tbbdfname, compression='gzip') #can't allocate mem
# analyzed_rptr_dfs[0].to_csv(lib1)
# analyzed_rptr_dfs[1].to_csv(lib2)
# adcounts_df_clean.to_csv(lib3)